In [ ]:
from pathlib import Path

import duckdb

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
CSV = ROOT / "dados/raw/eleitorado/2026/perfil_eleitorado_2026/perfil_eleitorado_2026_BRASIL.csv"
con = duckdb.connect()
con.execute("LOAD encodings")
con.sql(
    f"CREATE OR REPLACE VIEW dados_raw AS SELECT * FROM read_csv('{CSV.as_posix()}', delim=';', header=true, encoding='cp1252', all_varchar=true, ignore_errors=true)"
)
con.sql(f"COPY dados_raw TO '{ROOT / 'dados' / 'processed' / 'eleitorado.parquet'}' (FORMAT PARQUET, COMPRESSION ZSTD)")
con.sql(f"CREATE VIEW dados AS SELECT * FROM '{ROOT / 'dados' / 'processed' / 'eleitorado.parquet'}'")

In [ ]:
con.sql("DESCRIBE dados").df()

In [ ]:
con.sql("SUMMARIZE dados").df()

In [ ]:
con.sql("EXPLAIN SELECT * FROM dados").df()

In [ ]:
con.sql("SHOW ALL TABLES")

In [ ]:
con.sql("SELECT DISTINCT DT_GERACAO FROM dados").df()